# CIFAR10 Semantic Consolidation Continual DiT CLF

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/files/notebooks/CIFAR10%20Semantic%20Consolidation%20Continual%20DiT%20CLF.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Ffiles%2Fnotebooks%2FCIFAR10%2520Semantic%2520Consolidation%2520Continual%2520DiT%2520CLF.ipynb)
[![Launch Binder](https://img.shields.io/badge/launch-binder-F5793A?logo=jupyter&logoColor=white)](https://mybinder.org/v2/gh/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/main?urlpath=lab%2Ftree%2Ffiles%2Fnotebooks%2FCIFAR10%20Semantic%20Consolidation%20Continual%20DiT%20CLF.ipynb)
[![Open in Studio Lab](https://img.shields.io/badge/Open_in-Studio_Lab-232F3E)](https://studiolab.sagemaker.aws/import/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/files/notebooks/CIFAR10%20Semantic%20Consolidation%20Continual%20DiT%20CLF.ipynb)

Select a GPU runtime in Colab or Kaggle; enable Internet in Kaggle. Binder is suitable for inspecting the notebook. For Studio Lab, set `RUNTIME = "studiolab"` in the first code cell and use a GPU session for training.

These buttons open the published GitHub `main` version. Publish this notebook and its project setup files before using the links, or upload the notebook directly to Colab/Kaggle while it remains local. The first code cell prepares the project runtime; if it requests a kernel restart, restart and run the cells again.

One joint DiT learns to denoise and classify CIFAR-10, two new classes at a time. The previous joint DiT generates replay and teaches the next student; a separate U-Net supplies current-task denoising targets. After joint training, semantic acquisition and consolidation update the classifier's semantic representation.

This notebook uses our strongest **retention-focused** candidate: old-class CE weight **10**, classifier KD coefficient **0.0043**, and **DDIM200 / CFG1** replay. Its completed two-task development run reached **64.9% mean accuracy**, **69.2% old-task accuracy**, and **17.9 percentage points forgetting** (seed 101). These are historical screening results, not five-task results. The cells below train a fresh five-task model; their outputs are intentionally empty until execution.

In [ ]:
"""Prepare a notebook checkout and runtime before importing scientific packages.

This canonical first-cell source is copied into maintained notebooks and generated
HPO notebooks. Edit REVISION/RUNTIME/CUDA before execution when needed. Existing
checkouts are reused without fetching; only missing setup code is downloaded.
ROOT is the selected absolute pathlib.Path; RUNTIME_PACKAGES maps package names
to installed version strings. Preparation may clone/install under the selected
policy and changes cwd/import paths through files.notebooks.init.prepare_notebook.
"""
from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"
RUNTIME = "auto"
CUDA = None

_locations = (Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME, 
              Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME)
_initializer = next((path / "files" / "notebooks" / "init.py" for path in _locations
                     if (path / "files" / "notebooks" / "init.py").is_file()), None)
_url = f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/{REVISION}/files/notebooks/init.py"
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, 
    repository=REPOSITORY, 
    revision=REVISION, 
    runtime=RUNTIME, 
    cuda=CUDA
)

In [ ]:
from common import utils


utils.init()

## Experiment settings

We keep the same model across five tasks and expand its class vocabulary as classes arrive. Each task has 100 joint-training epochs on 2,500 current real images. From task 2 onward, the student also sees 500 generated old images in total. Old real images are not stored.

Run this notebook on a remote GPU with TensorFlow 2.20.0 / Keras 3.11.2 and this notebook's accompanying code changes. The model seed is 101; the data split and fixed class order use seed 42.

In [ ]:
import tensorflow as tf
from common.runtime import configure_runtime, derive_seed


epochs = 100
batch_size = 128
seed = 101
data_seed = 42
class_order = [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
task_groups = [[7, 3], [2, 8], [5, 6], [9, 4], [0, 1]]
current_examples = 2500
replay_examples = 500
replay_steps = 200
replay_cfg = 1.0
output_dir = ROOT / "files/results/cifar10_semantic_continual_dit_clf_seed101"

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("Run the training cells on a remote GPU runtime.")
configure_runtime(dtype_policy="float32", deterministic_ops=True, seed=seed)

## CIFAR10 data

The learner loads CIFAR-10 and selects each class pair in turn. Ten percent of the training split is held out for development evaluation. The official test split is evaluated separately after each completed task; neither split supplies gradient updates. The class order was randomized once and is fixed below.

Images stay in raw pixel coordinates at the loader and replay boundaries. The diffusion wrapper alone maps them to [-1, 1] for training and back to [0, 255] for generated replay. This avoids normalizing generated images twice.

In [ ]:
import pandas as pd
from common.dataloader import load_cifar10


dataset_settings = {
    "validation_ratio": 0.1, 
    "preprocess": None, 
    "onehot_labels": False, 
    "features_path": None, 
    "seed": data_seed
}
class_names = [
    "airplane", "automobile", "bird", "cat", "deer", 
    "dog", "frog", "horse", "ship", "truck"
]
pd.DataFrame({
    "Task": range(1, 6), 
    "Class IDs": task_groups, 
    "Classes": [", ".join(class_names[c] for c in group) for group in task_groups]
})

## Current-task denoising teacher

The U-Net supplies a denoising target for current classes. It is trained for 100 epochs on all 9,000 current-task training images, before fitting the student. Its own optimizer and expanding vocabulary persist across tasks. It does not generate old-class replay; that comes from the previous joint DiT.

In [ ]:
from diffusion.models.convolution.unet import UNet
from diffusion.models.wrapper.diffusion_model import DiffusionModel
from tensorflow.keras import optimizers


configure_runtime(
    dtype_policy="float32", 
    deterministic_ops=True, 
    seed=derive_seed(seed, "specialist_teacher", "noise")
)
unet = UNet(
    image_size=32, channels=3, num_classes=None, 
    widths=[64, 128, 256], block_depth=3, 
    bottleneck_width=384, bottleneck_depth=2, 
    image_embedding_dim=21, time_embedding_dim=22, label_embedding_dim=21, 
    activation_func="gelu", final_activation_func="linear", dropout_rate=0.0, 
    downsampling_method="avg_pooling", 
    upsampling_method="cnn_transpose", upsampling_interpolation="bilinear", 
    use_batch_norm=True, use_skip_connections=True, use_cfg=True, timesteps=1000
)
noise_teacher = DiffusionModel(
    network=unet, use_ema=False, 
    preprocess_type="standardize", scheduler_name="clipped_cosine", 
    modify_first_t=True, p_uncond=0.2, train_noisified_max_timesteps=1000, 
    noise_loss_coef=1.0, noise_distil_loss_coef=0.0, image_loss_coef=0.0, 
    swap_noise_image=False, trainable_teacher=False, defer_teacher=False, 
    test_network_name="raw", test_steps=50, test_cfg_scale=3.0, test_eta=0.0, 
    test_noisified_min_timesteps=0, test_noisified_max_timesteps=0
)
noise_teacher.compile(
    optimizer=optimizers.AdamW(
        learning_rate=0.0009232173954634668, 
        weight_decay=1.0569549424860102e-5
    ), 
    loss="mse"
)

## DiT denoiser and classifier

A 2-by-2 convolutional patch embedding feeds eight diffusion transformer blocks of width 128. Four classifier blocks read the shared representation. Both use four attention heads. The classifier has a linear hidden projection, dropout 0.5, and no DropPath. The class vocabulary starts empty and grows with the tasks.

Classification uses a null class condition, so the label being predicted is not supplied as an input. The same raw network handles classification and replay generation. EMA and ensemble prediction are disabled.

In [ ]:
from diffusion.models.transformer.di_t_classifier import DiTClassifier
from diffusion.models.wrapper.diffusion_classifier import DiffusionClassifier


configure_runtime(dtype_policy="float32", deterministic_ops=True, seed=seed)
vit = DiTClassifier(
    image_size=32, channels=3, num_classes=None, 
    patch_size=2, patchify_with_cnn=True, 
    depth=8, dim=128, dim_forced=True, 
    mha_num_heads=4, vit_block_mlp_ratio=4.0, droppath_rate=0.0, 
    clf_depth=4, clf_mha_num_heads=4, 
    clf_vit_block_mlp_ratio=4.0, clf_droppath_rate=0.0, 
    classifier_dropout_rate=0.5, classifier_mlp_activation_func="linear", 
    classifier_mlp_ratio=1, classifier_only_cls_token=True, 
    classifier_only_distil_token=True, clf_cls_token_type="new_weight", 
    clf_connection_ids_dict={-1: [-1]}, feature_aggregation_ids_dict={1: [-1]}, 
    force_global_avg_pooling=True, conds_merger_type="add", 
    patches_pos_embed_type="2d_sincos", patches_pos_merger_type="add", 
    timesteps=1000, use_cfg=True, use_refiner_cnn=False, 
    seed=seed
)

## Joint training and replay settings

The joint objective combines noise MSE (1.0), classification CE (0.00215), classifier distillation (0.0043), and denoiser distillation (0.5). Within the primary classification CE, old-class examples receive weight 10 and current-class examples weight 1; the weighted loss is normalized by the sum of weights. This does not multiply semantic CE or KD losses.

Classifier training and classifier KD use noisy inputs with timesteps below 64. Inference uses clean images. Soft classifier KD uses temperature 2 and old-class support, supplied by the frozen previous joint model. Current-task classifier-teacher weight is zero.

Replay uses 200 DDIM steps, guidance scale 1, and eta 0. The 500-image budget is shared across all old classes, rather than allocated to each class.

In [ ]:
model = DiffusionClassifier(
    network=vit, use_ema=False, 
    preprocess_type="standardize", scheduler_name="clipped_cosine", 
    modify_first_t=True, p_uncond=0.05, train_noisified_max_timesteps=1000, 
    noise_loss_coef=1.0, noise_distil_loss_coef=0.5, image_loss_coef=0.0, 
    swap_noise_image=False, 
    clf_loss_coef=0.00215, clf_old_class_loss_weight=10.0, 
    clf_distil_loss_coef=0.0043, clf_distil_type="soft", 
    clf_distil_temperature=2.0, clf_distil_scope="old_classes", 
    clf_train_class_input_type="null_class_only", 
    clf_train_noisy_input_type="noisy", clf_train_batch_fraction=0.0, 
    clf_train_noisified_max_timesteps=64, clf_test_noisified_max_timesteps=0, 
    clf_distil_noisy_input_type="noisy", 
    clf_distil_train_noisified_max_timesteps=64, 
    clf_distil_test_noisified_max_timesteps=0, 
    clf_acc_coef=1.0, clf_distil_acc_coef=0.0, 
    mask_by_nulls=False, mask_by_t_threshold=False, 
    previous_teacher_clf_loss_weight=1.0, current_teacher_clf_loss_weight=0.0, 
    previous_teacher_noise_loss_weight=1.0, current_teacher_noise_loss_weight=1.0, 
    teacher_training="each_task", teacher_dynamic_classes=True, 
    trainable_teacher=True, defer_teacher=True, 
    test_network_name="raw", test_steps=replay_steps, 
    test_cfg_scale=replay_cfg, test_eta=0.0, 
    test_noisified_min_timesteps=0, test_noisified_max_timesteps=0, 
    seed=seed
)

## Optimizer

The joint model uses Adam with a constant learning rate and clipnorm 5. The U-Net has its own optimizer above. Semantic phases use their separate learning rate below. Early stopping is disabled so each task receives its stated epoch budget.

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0009430051881577431, clipnorm=5.0), 
    loss="mse"
)

## Semantic consolidation

After joint fitting, 100 acquisition updates learn modulation states and 100 consolidation updates align semantic features. These are optimizer updates, not epochs. Consolidation changes the classifier projection/head while keeping the shared diffusion backbone fixed. Learned modulators are retained between tasks.

We use four TMCL-style augmented views and clean semantic features. Semantic CE has weight 1 on task 1 and 0 on later tasks; alignment stays active. Joint classification CE remains active on every task. This is the project's joint-DiT adaptation, not an exact reproduction of TMCL.

In [ ]:
from semantic_consolidation.config import RouteSettings
from semantic_consolidation.controller import RouteController
from semantic_consolidation.model import adapt_model


semantic_settings = RouteSettings(
    condition="learned", acquisition_objective="contrastive", 
    consolidation_scope="semantic", 
    acquisition_steps=100, consolidation_steps=100, 
    batch_size=32, learning_rate=0.001, temperature=0.1, 
    alignment_weight=1.0, orthogonality_weight=1.0, 
    task1_ce_weight=1.0, ce_weight=0.0, 
    image_augmentation="tmcl", augmentation_views=4, 
    acquisition_noise_level=0, ce_noise_level=0, noise_levels=[0], 
    gain_limit=1.0, bias_limit=1.0, modulation_init_std=0.02, 
    reliability="uniform", reliability_floor=0.05, retain_modulators=True, 
    probe_batches=4, probe_max_gates=16, checkpoint_interval=0, 
    seed=seed
)
route_controller = RouteController(semantic_settings)
model = adapt_model(model, route_controller)

## Continual training

From task 2 onward, the learner first generates old-class replay from the previous joint DiT. It fits the current noise teacher, trains the joint student, performs semantic acquisition/consolidation, and evaluates all seen classes. Task identity is not supplied to the classifier at evaluation. The completed joint model then becomes the next frozen teacher and replay generator.

The direct API's `legacy` evaluation mode reports both development and official test matrices. This is a new five-task presentation run. To keep the official test split locked during further tuning, change only `experiment_phase` to `development`; the historical 64.9% score used that development protocol.

In [ ]:
from common.learner import continually_learn


output_dir.mkdir(parents=True, exist_ok=True)
results = continually_learn(
    class_num=10, load_dataset_fn=load_cifar10, 
    load_dataset_fn_kwargs=dataset_settings, 
    class_order=class_order, task_groups=task_groups, task_size=2, 
    class_order_mode="fixed", task_order_mode="fixed", 
    generative_model=model, noise_teacher_network=noise_teacher, 
    use_generative_model_classifier=True, train_classifier_separately=False, 
    keep_same_model=True, remove_prev_classes=True, use_loaded_opt=True, 
    use_buffer=False, use_generative_replay=True, use_distillation=True, 
    dual_teacher_distillation=False, current_teacher_init="fresh", 
    replay_budget_mode="fixed_total", replay_old_examples=replay_examples, 
    replay_current_examples=current_examples, replay_selection="uniform", 
    replay_candidate_multiplier=1, replay_cache_mode="off", 
    generative_model_kwargs={"train_num": -1, "samples_per_class": 0}, 
    snapshot_network_name="raw", 
    epochs=epochs, batch_size=batch_size, shuffle_buffer=10000, 
    callback_patience=0, use_valset=True, 
    use_ensemble_accuracy=False, evaluate_ensemble_accuracy=False, 
    save_task_checkpoints=True, checkpoint_dir=str(output_dir / "checkpoints"), 
    experiment_phase="legacy", 
    plot_results=False, show_generated_images=False, show_network_summary=False, 
    return_details=True, deterministic_ops=True, dtype_policy="float32", 
    dataset_seed=data_seed, 
    verbose=2, seed=seed
)

## Save weights and semantic state

The learner saves a recovery checkpoint after every completed task. We also save the final model weights and the semantic records/modulators. Keep the task checkpoints for resumption; a weights file alone does not contain the optimizer and continual-learning state.

In [ ]:
model = results["generative_model"]
model.save_weights(str(output_dir / "final.weights.h5"))
route_controller.save(output_dir / "semantic")

## Continual evaluation

Rows are completed training tasks; columns are evaluation tasks. Each score uses the full seen-class classifier, even when evaluating one task's images. Blank cells denote classes that have not arrived yet. Values are percentages.

Final average accuracy averages the last row. Average incremental accuracy averages seen-task accuracy over the learning sequence. Forgetting measures each old task's drop from its previous best accuracy; backward transfer compares final accuracy with accuracy when that task was first learned. We use the learner's own metric implementation.

In [ ]:
from IPython.display import display


task_labels = [f"Task {i}" for i in range(1, len(task_groups) + 1)]
development_matrix = pd.DataFrame(
    results["validation_accuracy_matrix"], index=task_labels, columns=task_labels
)
development_matrix.to_csv(output_dir / "development_accuracy_matrix.csv")
print("Development accuracy (%)")
display((100 * development_matrix).round(2))

if results["test_evaluated"]:
    test_matrix = pd.DataFrame(
        results["ordinary_accuracy_matrix"], index=task_labels, columns=task_labels
    )
    test_matrix.to_csv(output_dir / "test_accuracy_matrix.csv")
    print("Official test accuracy (%)")
    display((100 * test_matrix).round(2))

In [ ]:
metric_columns = {"Development": results["validation_continual_metrics"]}
if results["test_evaluated"]:
    metric_columns["Official test"] = results["continual_metrics"]
metrics = pd.DataFrame(metric_columns)
metrics.to_csv(output_dir / "continual_metrics.csv")
metrics.to_json(output_dir / "continual_metrics.json", indent=2)
print("Accuracy in percent; forgetting and backward transfer in percentage points")
(100 * metrics).round(2)

## Learning across tasks

This small plot shows how old-task retention and current-task learning evolve. It uses official test scores when available, or development scores in development mode.

In [ ]:
trajectory = pd.DataFrame({
    "Old tasks": results["old_task_accuracy"], 
    "Current task": results["new_task_accuracy"]
}, index=range(1, len(task_groups) + 1))
trajectory.to_csv(output_dir / "old_new_accuracy.csv")
ax = (100 * trajectory).plot(marker="o", ylim=(0, 100), xticks=range(1, 6))
ax.set_xlabel("Completed task")
ax.set_ylabel("Accuracy (%)")
ax.figure.savefig(output_dir / "continual_accuracy.png", bbox_inches="tight")

## Class-conditional samples

Finally, show one image per class from the trained joint DiT using the replay sampler. Internal class IDs follow the learning order, so we label the grid accordingly. These ten images are a visual check, not a sample-quality estimate.

In [ ]:
samples = model.sample(
    network_name="raw", labels=None, 
    steps=replay_steps, scale=replay_cfg, eta=0.0, 
    verbose=0, seed=seed
)
utils.plot_images(
    samples, row=2, col=5, 
    save_path=output_dir / "final_samples.png", 
    titles=[class_names[c] for c in class_order], value_range=(0, 255)
)

## Settings provenance

This recipe is New5 worker 008 (oldCE10 / KD4 / DDIM200 / CFG1 / replay500). The recorded endpoint was two-task development accuracy: old 69.2%, new 60.6%, mean 64.9%, forgetting 17.9 points. Worker 004 had a slightly higher mean (65.3%) but much poorer old-task retention (43.6%); that is why this notebook uses worker 008. Neither result establishes five-task or three-seed performance.

Source records: [sampler analysis](../results/fleet_design_20261014_v1/new5_native_noise_roles_evidence_v3/parent_sampler_analysis.json), [candidate settings](../results/fleet_design_20261014_v1/new5_native_weighted_replay_quality_v2/selection.json), and [campaign preparation](../results/fleet_design_20261014_v1/native_schedule_v1/prepare_new5_weighted_quality.py). The original remote checkpoints were lost; this notebook trains from a fresh initialization.